Chiến lược xử lý:
Vấn đề độ dài: Các model BERT thường chỉ nhận tối đa 512 tokens (khoảng 300-400 từ). Tin tức của bạn rất dài.

Giải pháp: Tin tức tài chính thường "giật tít" và đưa thông tin quan trọng nhất ở đoạn đầu (Title + Sapô). Chúng ta sẽ nối Tiêu đề + 200 từ đầu tiên của nội dung để đưa vào model. Cách này đảm bảo độ chính xác cao nhất mà không bị lỗi độ dài.

Thang điểm: Model sẽ trả về POS (Tích cực), NEG (Tiêu cực), NEU (Trung lập). Ta sẽ quy đổi:

POS -> 1

NEU -> 0

NEG -> -1

In [1]:
import pandas as pd
import torch
from transformers import pipeline

# 1. Cấu hình thiết bị
device = 0 if torch.cuda.is_available() else -1
print(f"Đang chạy trên: {'GPU' if device == 0 else 'CPU'}")

# 2. Khởi tạo Model Pre-trained Tiếng Việt (PhoBERT Sentiment)
# Model này đã được train sẵn để nhận diện: NEG (Tiêu cực), POS (Tích cực), NEU (Trung lập)
print("Đang tải model...")
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="wonrax/phobert-base-vietnamese-sentiment",
    tokenizer="wonrax/phobert-base-vietnamese-sentiment",
    device=device,
    max_length=512,
    truncation=True,
    framework="pt"
)

# 3. Giả lập dữ liệu 
# Lưu ý: Cột content nên được xử lý sơ bộ (loại bỏ HTML tag nếu có)
data = [
    {"ticker": "MWG", "year": 2024, "title": "TTC Land vượt kế hoạch nửa đầu năm", "content": "TTC Land vượt kế hoạch nửa đầu năm"},
    {"ticker": "MWG", "year": 2024, "title": "Bách Hoá Xanh tuyên bố hoà vốn sau nhiều năm lỡ hẹn", "content": "CTCP Đầu tư Thế Giới Di Động (MWG) vừa công bố kết quả... Bách Hoá Xanh bất ngờ tuyên bố đã đạt mục tiêu hòa vốn..."},
    {"ticker": "MWG", "year": 2023, "title": "Thế giới Di động sa thải 13.000 người", "content": "Lợi nhuận thấp nhất từ trước đến nay... lợi nhuận sau thuế ghi nhận hơn 21 tỷ đồng, giảm 98,5%..."}
]
df_news = pd.DataFrame(data)

# --- HÀM XỬ LÝ CHÍNH ---
def calculate_sentiment(row):
    # 1. Kết hợp Tiêu đề + Nội dung (Ưu tiên tiêu đề vì nó chứa sentiment mạnh nhất)
    full_text = str(row['title']) + ". " + str(row['content'])[:1000]

    try:
        # Gọi model
        result = sentiment_analyzer(full_text)[0]
        label = result['label']
        score = result['score'] # Độ tin cậy của model (0.0 -> 1.0)

        # 2. Quy đổi sang điểm số (-1, 0, 1)
        if label == 'POS':
            return 1, score
        elif label == 'NEG':
            return -1, score
        else: # NEU
            return 0, score
    except Exception as e:
        print(f"Lỗi dòng: {row['title'][:20]}... - {e}")
        return 0, 0

# 4. Áp dụng cho toàn bộ dữ liệu
print("Đang tính toán Sentiment (có thể mất thời gian nếu dữ liệu lớn)...")

# Chạy hàm apply
df_news[['sentiment_label', 'confidence']] = df_news.apply(
    lambda row: pd.Series(calculate_sentiment(row)), axis=1
)

# 5. Tổng hợp điểm Sentiment theo Năm và Mã CK (Aggregating)
# Logic: Tính trung bình cộng sentiment của tất cả bài báo trong năm đó
df_final_sentiment = df_news.groupby(['ticker', 'year'])['sentiment_label'].mean().reset_index()
df_final_sentiment.rename(columns={'sentiment_label': 'SEN'}, inplace=True)

# Hiển thị kết quả
print("\n--- KẾT QUẢ CHI TIẾT TỪNG BÀI BÁO ---")
print(df_news[['ticker', 'year', 'title', 'sentiment_label', 'confidence']])

print("\n--- KẾT QUẢ CUỐI CÙNG (BIẾN SEN) ĐỂ MERGE VÀO FILE TÀI CHÍNH ---")
print(df_final_sentiment)

d:\anaconda\envs\nlp_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang chạy trên: CPU
Đang tải model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 28716.37it/s]


Đang tính toán Sentiment (có thể mất thời gian nếu dữ liệu lớn)...

--- KẾT QUẢ CHI TIẾT TỪNG BÀI BÁO ---
  ticker  year                                              title  \
0    MWG  2024                 TTC Land vượt kế hoạch nửa đầu năm   
1    MWG  2024  Bách Hoá Xanh tuyên bố hoà vốn sau nhiều năm l...   
2    MWG  2023              Thế giới Di động sa thải 13.000 người   

   sentiment_label  confidence  
0              1.0    0.948037  
1              1.0    0.791066  
2             -1.0    0.987547  

--- KẾT QUẢ CUỐI CÙNG (BIẾN SEN) ĐỂ MERGE VÀO FILE TÀI CHÍNH ---
  ticker  year  SEN
0    MWG  2023 -1.0
1    MWG  2024  1.0


In [2]:
import pandas as pd
import torch
import re
from transformers import pipeline
from tqdm import tqdm
from sqlalchemy import create_engine, text
import pymysql

# ================= THIẾT BỊ =================
if torch.backends.mps.is_available():
    device = "mps"
    print("Apple Silicon (MPS)")
elif torch.cuda.is_available():
    device = 0
    print("NVIDIA GPU")
else:
    device = -1
    print("CPU")

# ================= LOAD MODEL =================
print("Loading PhoBERT sentiment model...")
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="wonrax/phobert-base-vietnamese-sentiment",
    tokenizer="wonrax/phobert-base-vietnamese-sentiment",
    device=device,
    max_length=256,
    truncation=True
)

CPU
Loading PhoBERT sentiment model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 28871.75it/s]


In [3]:
res = sentiment_analyzer("Doanh thu VNM tăng 40%, lợi nhuận cao hơn kỳ vọng")
res2 = sentiment_analyzer("Doanh thu VNM tăng 40%")
print(res)
print(res2)

[{'label': 'POS', 'score': 0.9068813323974609}]
[{'label': 'POS', 'score': 0.8407805562019348}]


In [4]:
res = sentiment_analyzer("Vietnam Airlines công bố đã chấm dứt chuỗi 4 năm liên tiếp lỗ với mức doanh thu và lợi nhuận kỷ lục.")
print(res)

[{'label': 'NEU', 'score': 0.853914201259613}]


In [5]:
res = sentiment_analyzer("Giảm sàn 4/5 phiên gần nhất, vốn hóa \"bốc hơi\" 1 tỷ USD trong vòng 3 tuần, điều gì đang xảy ra với cổ phiếu Vietnam Airlines?")
print(res)

[{'label': 'NEG', 'score': 0.9515764713287354}]


In [6]:
res = sentiment_analyzer("Cổ phiếu HVN đã trải qua giai đoạn tăng nóng hơn 170% trong vòng 3 tháng để lên mức cao nhất 6 năm. Do vậy, áp lực chốt lời của nhà đầu tư là không thể tránh khỏi. Tiếp tục là một phiên giao dịch \"đáng quên\" với cổ đông nắm giữ cổ phiếu HVN của Vietnam Airlines. Sau kỳ thăng hoa trên đỉnh dài hạn, cổ phiếu HVN bất ngờ chứng kiến nhịp điều chỉnh mạnh kể từ đầu tháng 7 tới nay. Chốt phiên 22/7, thị giá HVN chìm trong sắc \"xanh lơ\", giảm hết biên độ về mức giá 24.350 đồng/cp với lượng dư bán sàn gần 4 triệu đơn vị. Kể từ đỉnh hồi đầu tháng 7 (36.400 đồng/cp), thị giá cổ phiếu hàng không này đã bốc hơi tới 33%. Vốn hóa thị trường cũng bị thổi bay tới 26.700 tỷ đồng (~1,05 tỷ USD) chỉ trong khoảng 3 tuần giao dịch, ghi nhận còn 53.920 tỷ đồng.")
print(res)

[{'label': 'POS', 'score': 0.9634743332862854}]


In [7]:
import torch
from transformers import pipeline

# ================= 1. KHỞI TẠO MODEL =================
device = "mps" if torch.backends.mps.is_available() else (0 if torch.cuda.is_available() else -1)
print(f"Loading model on: {device}...")

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="wonrax/phobert-base-vietnamese-sentiment",
    tokenizer="wonrax/phobert-base-vietnamese-sentiment",
    device=device,
    max_length=256,
    truncation=True
)

# ================= 2. ĐỊNH NGHĨA BỘ RULE=================
DISCLOSURE_KEYWORDS = [
    "báo cáo tài chính", "bctc", "kết quả kinh doanh", "kqkd",
    "tổng hợp kết quả", "báo cáo hợp nhất",
    "thông báo", "cbtt", "gdkhq",
    "hủy niêm yết", "huỷ niêm yết", "chuyển sàn",
    "kết quả quý", "quý i", "quý ii", "quý iii", "quý iv",
    "6 tháng", "9 tháng", "12 tháng", "cả năm"
]

INSIDER_KEYWORDS = [
    "người nội bộ", "đăng ký mua", "đăng ký bán",
    "thoái vốn", "bán cổ phiếu", "mua cổ phiếu"
]

INDUSTRY_KEYWORDS = [
    "ngành", "nhóm cổ phiếu", "lặng sóng",
    "toàn ngành", "thị trường", "cuộc chiến", "sóng gió"
]

def test_sentiment_pipeline(title, content):
    print(f"\n{'='*20} TESTING ARTICLE {'='*20}")
    print(f"Title: {title}")

    t_lower = title.lower()

    # --- RULE 1: DROP ---
    if any(k in t_lower for k in DISCLOSURE_KEYWORDS) or any(k in t_lower for k in INSIDER_KEYWORDS):
        return "RESULT: None (Dropped by Disclosure/Insider Rule)"


    # --- RULE 2: FORCE NEUTRAL ---
    if any(k in t_lower for k in INDUSTRY_KEYWORDS):
        return "RESULT: 0 (Forced Neutral by Industry/Macro Rule)"

    # --- RULE 3: PHO-BERT + DOUBLE VALIDATION ---
    def get_ai_prediction(text, window_name):
        res = sentiment_analyzer(text)[0]
        label, score = res["label"], res["score"]

        final_val = 0
        if score >= 0.7:
            if label in ["POS", "POSITIVE"]: final_val = 1
            elif label in ["NEG", "NEGATIVE"]: final_val = -1

        print(f"   [{window_name}] AI says: {label} (Score: {score:.4f}) -> Assigned: {final_val}")
        return final_val

    # Cửa sổ văn bản 50 vs 100 từ
    content_words = str(content).split()
    t50 = f"{title}. {' '.join(content_words[:50])}"
    t100 = f"{title}. {' '.join(content_words[:100])}"

    s1 = get_ai_prediction(t50, "50-word window")
    s2 = get_ai_prediction(t100, "100-word window")

    if s1 == s2:
        return f"RESULT: {s1} (Consistent AI Decision)"
    else:
        return "RESULT: 0 (Neutralized due to Inconsistency between 50/100 words)"

# ================= 3. CHẠY THỬ CÁC CASE KHÓ =================

# Case 1
case1_title = "Cổ phiếu HVN giảm sàn, bốc hơi hàng nghìn tỷ"
case1_content = "Thị trường chứng khoán chứng kiến nhịp điều chỉnh mạnh của HVN..."
print(test_sentiment_pipeline(case1_title, case1_content))

# Case 2
case2_title = "Cổ phiếu HVN thoát cảnh giảm sàn, lực cầu bắt đáy tăng mạnh"
case2_content = "Sau chuỗi ngày xanh lơ, cổ phiếu hàng không đã có dấu hiệu hồi phục..."
print(test_sentiment_pipeline(case2_title, case2_content))

# Case 3
case3_title = "Công bố BCTC Quý 4: Doanh thu tăng trưởng vượt bậc"
case3_content = "Công ty vừa công bố báo cáo tài chính hợp nhất..."
print(test_sentiment_pipeline(case3_title, case3_content))

# Case 4
case4_title = "Ông Đỗ Cao Bảo: FPT xây nhà máy AI không phải để cho oách"
case4_content = "Đại diện FPT khẳng định việc đầu tư vào Nhật Bản là chiến lược dài hạn..."
print(test_sentiment_pipeline(case4_title, case4_content))

Loading model on: -1...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 25124.57it/s]



==================== TESTING ARTICLE ====================
Title: Cổ phiếu HVN giảm sàn, bốc hơi hàng nghìn tỷ
   [50-word window] AI says: NEG (Score: 0.9858) -> Assigned: -1
   [100-word window] AI says: NEG (Score: 0.9858) -> Assigned: -1
RESULT: -1 (Consistent AI Decision)

==================== TESTING ARTICLE ====================
Title: Cổ phiếu HVN thoát cảnh giảm sàn, lực cầu bắt đáy tăng mạnh
   [50-word window] AI says: NEU (Score: 0.8086) -> Assigned: 0
   [100-word window] AI says: NEU (Score: 0.8086) -> Assigned: 0
RESULT: 0 (Consistent AI Decision)

==================== TESTING ARTICLE ====================
Title: Công bố BCTC Quý 4: Doanh thu tăng trưởng vượt bậc
RESULT: None (Dropped by Disclosure/Insider Rule)

==================== TESTING ARTICLE ====================
Title: Ông Đỗ Cao Bảo: FPT xây nhà máy AI không phải để cho oách
   [50-word window] AI says: NEG (Score: 0.5997) -> Assigned: 0
   [100-word window] AI says: NEG (Score: 0.5997) -> Assigned: 0
RESULT: 0 (C

In [8]:
# Case A: Tin số liệu thuần túy (Mục tiêu là muốn DROP)
case_a_title = "Doanh thu VNM tăng 40%"
case_a_content = "Công ty Cổ phần Sữa Việt Nam vừa công bố báo cáo hoạt động..."
print(test_sentiment_pipeline(case_a_title, case_a_content))

# Case B: Tin có sắc thái kỳ vọng (Mục tiêu là muốn lấy POS)
case_b_title = "Doanh thu VNM tăng 40% vượt mức kế hoạch năm"
case_b_content = "Đây là kết quả ấn tượng vượt xa các dự báo trước đó của giới phân tích..."
print(test_sentiment_pipeline(case_b_title, case_b_content))


==================== TESTING ARTICLE ====================
Title: Doanh thu VNM tăng 40%
   [50-word window] AI says: NEU (Score: 0.7738) -> Assigned: 0
   [100-word window] AI says: NEU (Score: 0.7738) -> Assigned: 0
RESULT: 0 (Consistent AI Decision)

==================== TESTING ARTICLE ====================
Title: Doanh thu VNM tăng 40% vượt mức kế hoạch năm
   [50-word window] AI says: POS (Score: 0.9775) -> Assigned: 1
   [100-word window] AI says: POS (Score: 0.9775) -> Assigned: 1
RESULT: 1 (Consistent AI Decision)


In [ ]:
import pandas as pd
import torch
import re
from transformers import pipeline
from tqdm import tqdm
from sqlalchemy import create_engine, text
import pymysql

# ================= CẤU HÌNH =================
DB_CONFIG = {
    "host": "127.0.0.1",
    "user": "root",
    "password": "password",
    "database": "cafef_news_db",
    "charset": "utf8mb4"
}

SOURCE_TABLE = "company_news"
TARGET_TABLE = "news_sentiment_final"

# ================= THIẾT BỊ =================
if torch.backends.mps.is_available():
    device = "mps"
    print("Apple Silicon (MPS)")
elif torch.cuda.is_available():
    device = 0
    print("NVIDIA GPU")
else:
    device = -1
    print("CPU")

# ================= LOAD MODEL =================
print("Loading PhoBERT sentiment model...")
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="wonrax/phobert-base-vietnamese-sentiment",
    tokenizer="wonrax/phobert-base-vietnamese-sentiment",
    device=device,
    max_length=256,
    truncation=True
)

# ================= DB =================
print("Reading data from MySQL...")
connection_str = (
    f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}/{DB_CONFIG['database']}?charset={DB_CONFIG['charset']}"
)
engine = create_engine(connection_str)

df = pd.read_sql(
    f"""
    SELECT id, ticker, year, title, content, url
    FROM {SOURCE_TABLE}
    """,
    engine
)
print(f"Loaded {len(df)} articles")

# ================= FILTER RULES =================

DISCLOSURE_KEYWORDS = [
    "báo cáo tài chính", "bctc", "kết quả kinh doanh", "kqkd",
    "tổng hợp kết quả", "báo cáo hợp nhất",
    "thông báo", "cbtt", "gdkhq",
    "hủy niêm yết", "huỷ niêm yết", "chuyển sàn",
    "kết quả quý", "quý i", "quý ii", "quý iii", "quý iv",
    "6 tháng", "9 tháng", "12 tháng", "cả năm"
]

INSIDER_KEYWORDS = [
    "người nội bộ", "đăng ký mua", "đăng ký bán",
    "thoái vốn", "bán cổ phiếu", "mua cổ phiếu"
]

INDUSTRY_KEYWORDS = [
    "ngành", "nhóm cổ phiếu", "lặng sóng",
    "toàn ngành", "thị trường", "cuộc chiến", "sóng gió"
]

TICKER_PATTERN = re.compile(r"^[A-Z]{2,5}[\s,:-]")
MULTI_TICKER_PATTERN = re.compile(r"[A-Z]{2,5}\s*&")

def is_disclosure(title):
    t = title.lower()
    return any(k in t for k in DISCLOSURE_KEYWORDS)

def is_insider(title):
    t = title.lower()
    return any(k in t for k in INSIDER_KEYWORDS)

def is_industry(title):
    t = title.lower()
    return any(k in t for k in INDUSTRY_KEYWORDS)

def is_ticker_spam(title):
    return bool(TICKER_PATTERN.match(title) or MULTI_TICKER_PATTERN.search(title))

# ================= SENTIMENT LOGIC =================

def build_text(title, content, n_words):
    title = title.strip()
    content = content.strip()
    lead = " ".join(content.split()[:n_words])
    return f"{title}. {lead}"

def predict_sentiment(text):
    r = sentiment_analyzer(text)[0]
    label = r["label"]
    score = r["score"]

    if score < 0.7:
        return 0

    if label in ["POS", "POSITIVE"]:
        return 1
    if label in ["NEG", "NEGATIVE"]:
        return -1
    return 0

def process_sentiment(row):
    title = str(row["title"]) if row["title"] else ""
    content = str(row["content"]) if row["content"] else ""

    if len(title) < 5:
        return None

    # ===== DROP =====
    if (
        is_disclosure(title)
        or is_insider(title)
        or is_ticker_spam(title)
    ):
        return None

    # ===== FORCE NEUTRAL =====
    if is_industry(title):
        return 0

    # ===== MODEL =====
    try:
        short_text = build_text(title, content, 50)
        long_text  = build_text(title, content, 100)

        s1 = predict_sentiment(short_text)
        s2 = predict_sentiment(long_text)

        if s1 == s2:
            return s1
        return 0
    except:
        return 0

print("Running sentiment analysis...")
tqdm.pandas()
df["sentiment_score"] = df.progress_apply(process_sentiment, axis=1)

# Loại bỏ các dòng bị đánh dấu None (do filter rules)
df = df[df["sentiment_score"].notna()].copy()
df["sentiment_score"] = df["sentiment_score"].astype(int)

# ================= SAVE (APPEND MODE) =================
print(f"Appending results to table `{TARGET_TABLE}`...")
final_df = df[["id", "year", "ticker", "url", "title", "sentiment_score"]]

try:
    # index=False để không lưu cột chỉ mục của pandas vào SQL
    final_df.to_sql(TARGET_TABLE, con=engine, if_exists="append", index=False)
    print(f"Successfully appended {len(final_df)} new rows.")

except Exception as e:
    print(f"Error during append: {e}")
    print("Mẹo: Nếu bị lỗi 'Duplicate entry', hãy kiểm tra xem ID đã tồn tại trong bảng chưa.")

# Thống kê tổng thể sau khi append
with engine.connect() as conn:
    total = conn.execute(text(f"SELECT COUNT(*) FROM {TARGET_TABLE}")).scalar()
    pos = conn.execute(text(f"SELECT COUNT(*) FROM {TARGET_TABLE} WHERE sentiment_score = 1")).scalar()
    neg = conn.execute(text(f"SELECT COUNT(*) FROM {TARGET_TABLE} WHERE sentiment_score = -1")).scalar()
    neu = conn.execute(text(f"SELECT COUNT(*) FROM {TARGET_TABLE} WHERE sentiment_score = 0")).scalar()

print(f"\n--- TOTAL TABLE SUMMARY ---")
print(f"Total rows in DB: {total}")
print(f"POS: {pos} | NEG: {neg} | NEU: {neu}")
print("DONE.")

d:\anaconda\envs\nlp_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU
Loading PhoBERT sentiment model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 25067.80it/s]


Reading data from MySQL...


Loaded 87643 articles
Running sentiment analysis...


100%|██████████| 87643/87643 [3:08:46<00:00,  7.74it/s]  


Appending results to table `news_sentiment_final`...
Successfully appended 63165 new rows.

--- TOTAL TABLE SUMMARY ---
Total rows in DB: 102190
POS: 41547 | NEG: 20649 | NEU: 39994
DONE.


In [38]:
import pandas as pd
import torch
import re
from transformers import pipeline
from tqdm import tqdm
from sqlalchemy import create_engine, text
import pymysql

# ================= CẤU HÌNH =================
DB_CONFIG = {
    "host": "127.0.0.1",
    "user": "root",
    "password": "password",
    "database": "cafef_news_db",
    "charset": "utf8mb4"
}

SOURCE_TABLE = "company_news"
TARGET_TABLE = "news_sentiment_final"

# ================= THIẾT BỊ =================
if torch.backends.mps.is_available():
    device = "mps"
    print("Apple Silicon (MPS)")
elif torch.cuda.is_available():
    device = 0
    print("NVIDIA GPU")
else:
    device = -1
    print("CPU")

# ================= LOAD MODEL =================
print("Loading PhoBERT sentiment model...")
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="wonrax/phobert-base-vietnamese-sentiment",
    tokenizer="wonrax/phobert-base-vietnamese-sentiment",
    device=device,
    max_length=256,
    truncation=True
)

# ================= DB =================
print("Reading data from MySQL...")
connection_str = (
    f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}/{DB_CONFIG['database']}?charset={DB_CONFIG['charset']}"
)
engine = create_engine(connection_str)

df = pd.read_sql(
    f"""
    SELECT id, ticker, year, title, content, url
    FROM {SOURCE_TABLE}
    """,
    engine
)
print(f"Loaded {len(df)} articles")

CPU
Loading PhoBERT sentiment model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 26632.60it/s]


Reading data from MySQL...
Loaded 87643 articles


In [39]:
SOURCE_TABLE = "company_news"
TARGET_TABLE = "news_sentiment_final"

In [40]:
# ================= TÙY CHỈNH SAMPLE =================
# Lựa chọn 1: Lấy 50 bài ngẫu nhiên (Random) để khách quan
QUERY_RANDOM = f"SELECT id, ticker, title, content FROM {SOURCE_TABLE} ORDER BY RAND() LIMIT 15"

# Lựa chọn 2: Lấy 50 bài của các năm gần đây (2023-2024) để mang tính thời sự
QUERY_RECENT = f"SELECT id, ticker, title, content FROM {SOURCE_TABLE} WHERE year IN (2010, 2011) LIMIT 10"

# Lựa chọn 3: Lấy các bài báo có chứa từ khóa nhạy cảm để test độ nhạy của 0.7
QUERY_SENSITIVE = f"""
    SELECT id, ticker, title, content FROM {SOURCE_TABLE}
    WHERE title LIKE '%lỗ%' OR title LIKE '%lãi%' OR title LIKE '%cảnh báo%'
    LIMIT 50
"""

print("Reading new sample data from MySQL...")
df = pd.read_sql(QUERY_RANDOM, engine)
print(f"Loaded {len(df)} sample articles.")

Reading new sample data from MySQL...
Loaded 15 sample articles.


In [41]:
# ================= TỪ ĐIỂN TÀI CHÍNH (LEXICON BASELINE) =================
lexicon_pos = ['lãi', 'tăng trưởng', 'vượt kế hoạch', 'tích cực', 'hợp tác', 'cải thiện', 'phục hồi', 'tăng']
lexicon_neg = ['lỗ', 'giảm', 'cảnh báo', 'kiểm soát', 'đình chỉ', 'vi phạm', 'sụt giảm', 'thua lỗ', 'kém']

def predict_lexicon(text):
    text = str(text).lower()
    pos_score = sum(1 for word in lexicon_pos if word in text)
    neg_score = sum(1 for word in lexicon_neg if word in text)
    if pos_score > neg_score: return 1
    if neg_score > pos_score: return -1
    return 0

# ================= PHOBERT LOGIC (COMPARE 0.5 VS 0.7) =================
def get_phobert_sentiment(text, threshold):
    res = sentiment_analyzer(text)[0]
    score = res["score"]
    label = res["label"]

    if score < threshold:
        return 0 # Trung lập nếu không đủ tin cậy

    if label in ["POS", "POSITIVE"]: return 1
    if label in ["NEG", "NEGATIVE"]: return -1
    return 0

In [43]:
tqdm.pandas()

# Tiền xử lý text (kết hợp Title + Content ngắn)
def prepare_text(row):
    title = str(row['title']).strip() if row['title'] else ""
    content = str(row['content']).strip() if row['content'] else ""
    return f"{title}. {' '.join(content.split()[:50])}"

df['full_text'] = df.apply(prepare_text, axis=1)

# Chạy dự báo
print("Đang chạy so sánh các phương pháp...")
df['lexicon_label'] = df['title'].apply(predict_lexicon)
df['phobert_05'] = df['full_text'].progress_apply(lambda x: get_phobert_sentiment(x, 0.5))
df['phobert_07'] = df['full_text'].progress_apply(lambda x: get_phobert_sentiment(x, 0.7))

# Hiển thị bảng đối chiếu 3 cột quan trọng nhất
pd.set_option('display.max_colwidth', 100)
comparison_table = df[['title', 'lexicon_label', 'phobert_05', 'phobert_07']].copy()

# Đổi số sang chữ
mapping = {1: "Tích cực", -1: "Tiêu cực", 0: "Trung lập"}
comparison_table['Lexicon (Baseline)'] = comparison_table['lexicon_label'].map(mapping)
comparison_table['PhoBERT (Ngưỡng 0.5)'] = comparison_table['phobert_05'].map(mapping)
comparison_table['PhoBERT (Ngưỡng 0.7)'] = comparison_table['phobert_07'].map(mapping)

# In ra các trường hợp có sự khác biệt để làm bằng chứng giải trình
diff_df = comparison_table[
    (comparison_table['lexicon_label'] != comparison_table['phobert_05']) |
    (comparison_table['lexicon_label'] != comparison_table['phobert_07']) |
    (comparison_table['phobert_05'] != comparison_table['phobert_07'])
]
print("\n--- CÁC TRƯỜNG HỢP CÓ SỰ KHÁC BIỆT GIỮA 3 PHƯƠNG PHÁP ---")
print(
    diff_df[
        [
            'title',
            'Lexicon (Baseline)',
            'PhoBERT (Ngưỡng 0.5)',
            'PhoBERT (Ngưỡng 0.7)'
        ]
    ].head(15)
)

Đang chạy so sánh các phương pháp...


100%|██████████| 15/15 [00:00<00:00, 16.99it/s]


--- CÁC TRƯỜNG HỢP CÓ SỰ KHÁC BIỆT GIỮA 3 PHƯƠNG PHÁP ---
                                                                                     title  \
1               Thêm một công ty họ Sông Đà “năm lần bảy lượt” khất trả cổ tức cho cổ đông   
2   Đất Xanh (DXG) muốn thoái sạch vốn tại Đầu tư LDG với giá không thấp hơn 6.000 đồng/cp   
4                      Công ty Sinh Thái đã chuyển nhượng hơn 34,6 triệu cổ phiếu Vingroup   
5                Cổ phiếu về đáy 3 năm, CII vẫn tính mang hơn 35 triệu cổ phiếu quỹ ra bán   
8                                   Bán Vinamilk, Nhà nước có ngay 60.000 tỷ đồng chi tiêu   
11                   Đô thị Sài Đồng dự chi 1.000 tỷ đồng trả cổ tức năm 2014 tỷ lệ 83,33%   
12                       VNC: Không thông qua phương án tăng vốn điều lệ lên 157,5 tỷ đồng   
13                        Chứng khoán sôi động, một loạt doanh nghiệp vốn nghìn tỷ lên sàn   
14                               CII vừa chuyển nhượng thêm 10 triệu cổ phiếu LGC cho MPTC   

